# 1. Train the Connect-4 agent

The agent learns only by playing against itself. Each **iteration**:

1. plays 40 games against itself, choosing moves with tree search (MCTS) guided by the network,
2. trains the network on those games: the *policy* learns to predict the search's moves, the *value* learns to predict who won,
3. every 5 iterations, plays 20 games against a random player and 20 against a greedy player to measure progress.

40 iterations take about **1 hour** on a normal CPU. A GPU is not needed: the network is small.
A checkpoint is saved after every iteration, so if Colab disconnects just run all cells again and training continues where it stopped.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/meg-x08/connect4-alphazero-xai.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Save checkpoints and results to Google Drive so they survive a disconnect
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/connect4-zero"
    REPO_DIR = "/content/repo"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
else:
    # Running locally from the notebooks/ folder
    REPO_DIR = os.path.abspath("..")
    WORK_DIR = REPO_DIR

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
CHECKPOINT_DIR = os.path.join(WORK_DIR, "checkpoints")
RESULTS_DIR = os.path.join(WORK_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("checkpoints:", CHECKPOINT_DIR)
print("results:    ", RESULTS_DIR)

In [ ]:
import torch
from connect4.train import train

torch.manual_seed(0)
net, log = train(
    save_dir=CHECKPOINT_DIR,
    iterations=40,
    games_per_iteration=40,
    num_simulations=50,
)

## Training curves

In [ ]:
import matplotlib.pyplot as plt

iterations = [e["iteration"] for e in log]
evals = [e for e in log if "win_rate_vs_random" in e]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(iterations, [e["policy_loss"] for e in log], label="policy loss")
ax1.plot(iterations, [e["value_loss"] for e in log], label="value loss")
ax1.set_xlabel("iteration")
ax1.set_title("Training loss")
ax1.legend()

ax2.plot([e["iteration"] for e in evals], [e["win_rate_vs_random"] for e in evals], "o-", label="vs random player")
ax2.plot([e["iteration"] for e in evals], [e["win_rate_vs_greedy"] for e in evals], "o-", label="vs greedy player")
ax2.set_ylim(0, 1.05)
ax2.set_xlabel("iteration")
ax2.set_title("Win rate (20 games each)")
ax2.legend()

fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "training.png"), dpi=150)
plt.show()